## 0. Importar Librerías y Cargar Datos

Importamos las librerías necesarias y cargamos los datasets desde la carpeta `datasets/`.

In [2]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
from pathlib import Path
import json
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import warnings

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Librerías importadas exitosamente")

✅ Librerías importadas exitosamente


In [3]:
# Definir ruta de la carpeta de datasets
datasets_folder = Path('datasets')

# Verificar que la carpeta exista
if not datasets_folder.exists():
    raise FileNotFoundError(f"La carpeta '{datasets_folder}' no existe. Ejecute primero el notebook eda.ipynb para generar los datasets.")

print("="*80)
print("CARGANDO DATASETS DESDE CARPETA LOCAL")
print("="*80)

# Cargar metadata
metadata_path = datasets_folder / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print(f"\n📋 Metadata cargada: {len(metadata['datasets'])} datasets disponibles")
    print(f"📅 Fecha de extracción: {metadata['fecha_extraccion']}")
else:
    print("\n⚠️ Archivo metadata.json no encontrado")
    metadata = None

# Mapeo de archivos CSV
dataset_files = {
    'delitos_bucaramanga': 'delitos_bucaramanga.csv',
    'info_delictiva_bucaramanga': 'info_delictiva_bucaramanga.csv',
    'delitos_sexuales': 'delitos_sexuales.csv',
    'violencia_intrafamiliar': 'violencia_intrafamiliar.csv',
    'hurto_modalidades': 'hurto_modalidades.csv'
}

# Cargar todos los datasets
dataframes = {}

for key, filename in dataset_files.items():
    filepath = datasets_folder / filename
    
    if filepath.exists():
        print(f"\n📂 Cargando: {filename}")
        df = pd.read_csv(filepath, encoding='utf-8')
        dataframes[key] = df
        print(f"   ✅ Cargado: {len(df):,} registros × {len(df.columns)} columnas")
        print(f"   📊 Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    else:
        print(f"\n⚠️ Archivo no encontrado: {filename}")

print(f"\n{'='*80}")
print(f"✅ CARGA COMPLETADA")
print(f"{'='*80}")
print(f"\n📦 Datasets cargados: {len(dataframes)}")
print(f"📊 Total de registros: {sum(len(df) for df in dataframes.values()):,}")

# Mostrar resumen
print(f"\n📋 RESUMEN DE DATASETS CARGADOS:\n")
summary_data = []
for key, df in dataframes.items():
    summary_data.append({
        'Dataset': key,
        'Registros': f"{len(df):,}",
        'Columnas': len(df.columns),
        'Memoria_MB': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

CARGANDO DATASETS DESDE CARPETA LOCAL

📋 Metadata cargada: 5 datasets disponibles
📅 Fecha de extracción: 2025-11-20T12:59:58.929861

📂 Cargando: delitos_bucaramanga.csv
   ✅ Cargado: 135,076 registros × 19 columnas
   ✅ Cargado: 135,076 registros × 19 columnas
   📊 Tamaño en memoria: 135.41 MB

📂 Cargando: info_delictiva_bucaramanga.csv
   📊 Tamaño en memoria: 135.41 MB

📂 Cargando: info_delictiva_bucaramanga.csv
   ✅ Cargado: 120,940 registros × 26 columnas
   ✅ Cargado: 120,940 registros × 26 columnas
   📊 Tamaño en memoria: 143.10 MB

📂 Cargando: delitos_sexuales.csv
   ✅ Cargado: 21,859 registros × 9 columnas
   📊 Tamaño en memoria: 10.31 MB

📂 Cargando: violencia_intrafamiliar.csv
   ✅ Cargado: 50,864 registros × 8 columnas
   📊 Tamaño en memoria: 18.17 MB

📂 Cargando: hurto_modalidades.csv
   ✅ Cargado: 1,445 registros × 9 columnas
   📊 Tamaño en memoria: 0.62 MB

✅ CARGA COMPLETADA

📦 Datasets cargados: 5
📊 Total de registros: 330,184

📋 RESUMEN DE DATASETS CARGADOS:

   📊 Tamañ

,Dataset,Registros,Columnas,Memoria_MB
0,delitos_bucaramanga,"135,076",19,135.41
1,info_delictiva_bucaramanga,"120,940",26,143.10
2,delitos_sexuales,"21,859",9,10.31
3,violencia_intrafamiliar,"50,864",8,18.17
4,hurto_modalidades,"1,445",9,0.62


# PIPELINES: LIMPIEZA Y PREPROCESAMIENTO
1. Elección de columnas categoricas
2. Creación del pipeline
3. Aplicación del pipeline en cada dataset

## Elección de columnas a tranformar
Se eligen las columnas categoricas a preprocesar
- Columnas pendientes de preprocesamiento: Longitud, latitud, codigo dane, fecha_hecho

In [4]:
columnas_por_dataset = {
    "delitos_bucaramanga": {
        "categoricas": ["armas_medios", "barrios_hecho","zona","nom_comuna","conducta","clasificaciones_delito","curso_de_vida","estado_civil_persona","genero","movil_agresor","movil_victima"],
        #"float": ["latitud", "longitud"]         
    },
    "info_delictiva_bucaramanga": {
        "categoricas": ["descripcion_conducta", "armas_medios","barrios_hecho","sexo","movil_victima","movil_agresor","delito_solo","tipolog_a","nom_com"],
        #"float": ["edad"]
    },
    "delitos_sexuales": {
        "categoricas": ["municipio", "armas_medios","genero","grupo_etario","delito"],
        "fechas": ["fecha_hecho"],
           },
    "violencia_intrafamiliar": {
        "categoricas": ["municipio", "armas_medios","genero","grupo_etario"],
        "fechas": ["fecha_hecho"]
    },
    "hurto_modalidades": {
        "categoricas": ["municipio", "armas_medios","genero","grupo_etario","tipo_de_hurto"],
        "fechas": ["fecha_hecho"]
    }
}

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class ObjectToFloatTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columnas):
        self.columnas = columnas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columnas:
            X[col] = (
                X[col]
                .astype(str)
                .str.replace(",", ".", regex=False)
                .astype(float)
            )
        return X


In [6]:
class DateSplitter(BaseEstimator, TransformerMixin):
    def __init__(self, columnas):
        self.columnas = columnas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columnas:
            X[col] = pd.to_datetime(X[col], format="%d/%m/%Y", errors="coerce")
            X[col + "_anio"] = X[col].dt.year
            X[col + "_mes"] = X[col].dt.month
            X[col + "_dia"] = X[col].dt.day
            X.drop(columns=[col], inplace=True)
        return X


In [7]:
def crear_pipeline_por_dataset(df, nombre_dataset):
    config = columnas_por_dataset[nombre_dataset]

    cols_cat = config["categoricas"]
    cols_fecha = config.get("fechas", [])
    cols_float = config.get("float", [])

    # Columnas numéricas restantes
    cols_num = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

    pipeline_pre = Pipeline(steps=[
        ("float_conv", ObjectToFloatTransformer(cols_float)),
        ("date_split", DateSplitter(cols_fecha))
    ])

    # OneHot solo a categóricas definidas por ti
    col_transform = ColumnTransformer(
        transformers=[
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols_cat),
        ],
        remainder="passthrough"
    )

    full_pipeline = Pipeline(steps=[
        ("prep", pipeline_pre),
        ("encode", col_transform)
    ])

    return full_pipeline


In [ ]:
datasets_transformados = {}

for nombre, df in dataframes.items():
    print(f"\n🔹 Procesando dataset: {nombre}")

    pipeline = crear_pipeline_por_dataset(df, nombre)
    datos_trans = pipeline.fit_transform(df)
    
    # Guardar
    datasets_transformados[nombre] = {
        "pipeline": pipeline,
        "datos": datos_trans
    }
    print(f"    ✔️ Dataset procesado. Nueva shape: {datos_trans.shape}")


print("\n✨ Todos los datasets fueron transformados correctamente.")



🔹 Procesando dataset: delitos_bucaramanga
    ✔️ Dataset procesado. Nueva shape: (135076, 656)

🔹 Procesando dataset: info_delictiva_bucaramanga
    ✔️ Dataset procesado. Nueva shape: (135076, 656)

🔹 Procesando dataset: info_delictiva_bucaramanga
    ✔️ Dataset procesado. Nueva shape: (120940, 466)

🔹 Procesando dataset: delitos_sexuales
    ✔️ Dataset procesado. Nueva shape: (120940, 466)

🔹 Procesando dataset: delitos_sexuales
    ✔️ Dataset procesado. Nueva shape: (21859, 134)

🔹 Procesando dataset: violencia_intrafamiliar
    ✔️ Dataset procesado. Nueva shape: (21859, 134)

🔹 Procesando dataset: violencia_intrafamiliar
    ✔️ Dataset procesado. Nueva shape: (50864, 110)

🔹 Procesando dataset: hurto_modalidades
    ✔️ Dataset procesado. Nueva shape: (1445, 102)

✨ Todos los datasets fueron transformados correctamente.
    ✔️ Dataset procesado. Nueva shape: (50864, 110)

🔹 Procesando dataset: hurto_modalidades
    ✔️ Dataset procesado. Nueva shape: (1445, 102)

✨ Todos los datasets